<a href="https://colab.research.google.com/github/amyylxn/Ontario-Price-Forecasting/blob/main/price_forecasting.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [6]:
import glob
import pandas as pd

# Only plain year files (no _v suffix, and not the rolling "most recent" file)
files = sorted(glob.glob(f'{DATA_DIR}/ieso/PUB_PriceHOEPPredispOR_[0-9][0-9][0-9][0-9].csv'))
files = [f for f in files if 2016 <= int(f[-8:-4]) <= 2025]

frames = []
for f in files:
    year = f[-8:-4]
    d = pd.read_csv(f, skiprows=3, usecols=['Date', 'Hour', 'HOEP'])
    print(year, d.shape)
    frames.append(d)

price = pd.concat(frames, ignore_index=True)
price['Date'] = pd.to_datetime(price['Date'])
price['timestamp'] = price['Date'] + pd.to_timedelta(price['Hour'] - 1, unit='h')
price = price.sort_values('timestamp').reset_index(drop=True)

print(price.shape)
price.head()

2016 (8784, 3)
2017 (8760, 3)
2018 (8760, 3)
2019 (8760, 3)
2020 (8784, 3)
2021 (8760, 3)
2022 (8760, 3)
2023 (8760, 3)
2024 (8784, 3)
2025 (2880, 3)
(81792, 4)


,Date,Hour,HOEP,timestamp
0,2016-01-01,1,-1.09,2016-01-01 00:00:00
1,2016-01-01,2,-2.41,2016-01-01 01:00:00
2,2016-01-01,3,-4.23,2016-01-01 02:00:00
3,2016-01-01,4,-4.4,2016-01-01 03:00:00
4,2016-01-01,5,-4.44,2016-01-01 04:00:00


In [7]:
# Duplicates and missing values
print(price['timestamp'].duplicated().sum(), "duplicate timestamps")
print(price['HOEP'].isna().sum(), "missing HOEP values")

# Gaps: every step between rows should be exactly 1 hour
gaps = price['timestamp'].diff().dropna()
print(gaps.value_counts())

# Overall range
print(price['timestamp'].min(), "to", price['timestamp'].max())

0 duplicate timestamps
0 missing HOEP values
timestamp
0 days 01:00:00    81791
Name: count, dtype: int64
2016-01-01 00:00:00 to 2025-04-30 23:00:00
